# Bayesian Ridge and Random Forest Regression using LOOCV

Bayesian Ridge and Random Forest regression models with Leave-One-Out Cross-Validation (LOOCV) were used for predicting the reactivity of 33 natural pozzolans (NPs).

This notebook contains the modeling pipeline:
1. Load and clean the dataset
2. Run BayesianRidge LOOCV with softplus link function
3. Run RandomForest LOOCV as baseline comparison
4. Generate parity plots for visual evaluation

In [ ]:
import sys
import os

# Ensure project root is in path (needed when running from notebooks/ folder)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

from config import (
    PATH_EXCEL, OUTPUT_DIR,
    PATH_CLEANED_DATA, PATH_LOOCV_BAYES,
    FEATURE_COLUMNS, TARGET_COLUMN, ID_COLUMN,
    RANDOM_STATE,
)
from models.bayesian_loocv import loocv_bayesian_softplus
from models.rf_loocv import loocv_random_forest

import warnings
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_STATE)

print('Libraries loaded')

## 1. Load and clean data

In [ ]:
df_raw = pd.read_excel(PATH_EXCEL)

# Create ID column if absent
if ID_COLUMN not in df_raw.columns:
    df_raw[ID_COLUMN] = np.arange(len(df_raw))

# Keep needed columns and drop rows with missing values
cols_needed = [ID_COLUMN] + FEATURE_COLUMNS + [TARGET_COLUMN]
df_clean = (
    df_raw[cols_needed]
    .dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])
    .copy()
)

print(f'Samples after cleaning: {len(df_clean)}')
df_clean.head()

In [ ]:
# Save cleaned dataset
df_clean.to_excel(PATH_CLEANED_DATA, index=False)
print(f'Cleaned data saved to {PATH_CLEANED_DATA}')

## 2. Bayesian Ridge + LOOCV (using softplus link)

In [ ]:
df_bayes = loocv_bayesian_softplus(df_clean, FEATURE_COLUMNS, TARGET_COLUMN)

# Save results
df_bayes.to_excel(PATH_LOOCV_BAYES, index=False)
print(f'Bayesian LOOCV results saved to {PATH_LOOCV_BAYES}')

## 3. Random Forest + LOOCV

In [ ]:
df_rf = loocv_random_forest(
    df_clean, FEATURE_COLUMNS, TARGET_COLUMN,
    n_estimators=100, max_depth=3, random_state=RANDOM_STATE,
)

## 4. Parity plots

In [ ]:
def parity_plot(y_true, y_pred, title, ax):
    """Draw a parity (actual vs predicted) scatter plot."""
    r2 = r2_score(y_true, y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    pad = 0.03 * (hi - lo)
    lo, hi = lo - pad, hi + pad

    ax.scatter(y_true, y_pred, s=50, alpha=0.9,
               facecolor='#4A90D9', edgecolor='black', linewidth=0.4,
               label='LOOCV points')
    ax.plot([lo, hi], [lo, hi], '--', color='gray', linewidth=1.5,
            label='Perfect prediction')
    ax.set_xlabel(f'Actual ({TARGET_COLUMN})', fontsize=11)
    ax.set_ylabel('Predicted', fontsize=11)
    ax.set_title(f'{title} (R2={r2:.3f})', fontsize=12)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.legend(loc='lower right', fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=150)

parity_plot(df_bayes['y_true'].values, df_bayes['y_pred'].values,
            'BayesianRidge LOOCV', axes[0])

parity_plot(df_rf['y_true'].values, df_rf['y_pred'].values,
            'RandomForest LOOCV', axes[1])

plt.tight_layout()
plt.show()